# Item-Based Collaborative Filtering

The idea: two articles are "similar" if the same users tend to click both of them. To recommend articles for a user, we look at what they have already clicked (their history) and rank candidates by how similar they are to those history items.

Unlike the most-popular baseline, this model is **personalised** — different users with different histories get different rankings.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize
from sklearn.metrics import roc_auc_score

TRAIN_BEHAVIORS = "../smallDataset/MINDsmall_train/behaviors.tsv"
DEV_BEHAVIORS   = "../smallDataset/MINDsmall_dev/behaviors.tsv"

COLS = ["impression_id", "user_id", "time", "history", "impressions"]

## Step 1 — Collect all user-article clicks from the training set

We treat any click as a positive interaction. A user "clicked" an article if:
- It appears in their **history** (past clicks), OR
- It appears in an **impression with label 1** (they clicked it when shown)

We gather every `(user, article)` pair across the training data.

In [2]:
train = pd.read_csv(TRAIN_BEHAVIORS, sep="\t", header=None, names=COLS)

user_clicks = defaultdict(set)

for _, row in train.iterrows():
    uid = row["user_id"]

    # Articles from history
    if isinstance(row["history"], str) and row["history"].strip():
        for nid in row["history"].split():
            user_clicks[uid].add(nid)

    # Clicked articles from impressions
    if isinstance(row["impressions"], str):
        for item in row["impressions"].split():
            nid, label = item.rsplit("-", 1)
            if label == "1":
                user_clicks[uid].add(nid)

print(f"Unique users with at least 1 click: {len(user_clicks)}")
print(f"Total (user, article) clicks: {sum(len(v) for v in user_clicks.values())}")

Unique users with at least 1 click: 50000
Total (user, article) clicks: 1148447


**What this tells us:**

We now have a list of every article each user has ever clicked. This is the raw data we use to figure out which articles "go together" — if two articles were clicked by many of the same users, we consider them similar.

## Step 2 — Build the sparse user-article interaction matrix

We turn the click data into a big matrix: rows are users, columns are articles, and a `1` means "this user clicked this article". Almost all entries are 0 (nobody clicks more than a tiny fraction of articles), so we use a **sparse matrix** from `scipy` to save memory.

We also build lookup dictionaries so we can convert between user/article IDs and matrix indices.

In [3]:
# Assign integer indices
user_to_idx    = {u: i for i, u in enumerate(user_clicks.keys())}
all_articles   = sorted({nid for clicks in user_clicks.values() for nid in clicks})
article_to_idx = {a: i for i, a in enumerate(all_articles)}

n_users    = len(user_to_idx)
n_articles = len(article_to_idx)

# Build coordinate lists for the sparse matrix
rows, cols = [], []
for uid, clicks in user_clicks.items():
    u_idx = user_to_idx[uid]
    for nid in clicks:
        rows.append(u_idx)
        cols.append(article_to_idx[nid])

data = np.ones(len(rows), dtype=np.float32)
interaction = csr_matrix((data, (rows, cols)), shape=(n_users, n_articles))

print(f"Interaction matrix shape: {interaction.shape}")
print(f"Non-zero entries: {interaction.nnz}")
print(f"Density: {100 * interaction.nnz / (n_users * n_articles):.4f}%")

Interaction matrix shape: (50000, 39865)
Non-zero entries: 1148447
Density: 0.0576%


**What this tells us:**

The matrix is huge but extremely sparse — only a tiny fraction of cells are non-zero. This is exactly why we use a sparse matrix: storing it densely would waste gigabytes of memory on zeros. Sparse storage keeps only the clicks we actually have.

## Step 3 — Normalise articles for cosine similarity

Cosine similarity between two articles is just the dot product of their L2-normalised click vectors. If we normalise once upfront, we can compute similarities later with a simple matrix multiplication.

We transpose the matrix so each **row** is an article (and columns are users). Then we L2-normalise each row.

In [4]:
# Article-as-row matrix (n_articles x n_users), L2-normalised per row
article_user = interaction.T.tocsr()
article_user_norm = normalize(article_user, norm="l2", axis=1, copy=False)

print(f"Article-user matrix shape: {article_user_norm.shape}")
print("Each row is now L2-normalised → dot products give cosine similarities.")

Article-user matrix shape: (39865, 50000)
Each row is now L2-normalised → dot products give cosine similarities.


**What this tells us:**

After normalisation, the dot product of two article rows equals their cosine similarity (a number between 0 and 1 for non-negative click data). We will use this to compute similarities between candidate articles and history articles on the fly.

We do **not** precompute the full article × article similarity matrix — that would be ~65k × 65k, way too large. Instead, we compute similarities on demand for each impression, which is fast because we only need a handful of rows at a time.

## Step 4 — Define the scoring function

For each impression:
1. Look up the rows of the user's history articles in the normalised matrix
2. Look up the rows of the candidate articles
3. Compute `candidates @ history.T` — a small matrix of cosine similarities
4. Average across history items → one score per candidate

**Cold-start handling:** if a candidate or history article was never seen in training, it gets similarity 0 (falls back to "unknown").

In [5]:
def cf_score(history: list, candidates: list) -> list:
    # Map history to known indices
    hist_idx = [article_to_idx[h] for h in history if h in article_to_idx]

    # If the user has no known history, we have nothing to compare against
    if not hist_idx:
        return [0.0] * len(candidates)

    hist_vecs = article_user_norm[hist_idx]  # (h, n_users)

    scores = []
    for cand in candidates:
        c_idx = article_to_idx.get(cand)
        if c_idx is None:
            scores.append(0.0)
            continue
        cand_vec = article_user_norm[c_idx]            # (1, n_users)
        sims = cand_vec @ hist_vecs.T                   # (1, h)
        sims = np.asarray(sims.todense()).ravel()
        scores.append(float(sims.mean()))

    return scores

## Step 5 — Evaluation

Same four metrics as the baseline so we can compare directly: AUC, MRR, nDCG@5, nDCG@10.

In [6]:
def dcg_at_k(relevance, k):
    relevance = np.array(relevance[:k], dtype=np.float32)
    if relevance.size == 0:
        return 0.0
    return float((relevance / np.log2(np.arange(2, relevance.size + 2))).sum())

def ndcg_at_k(relevance, k):
    ideal = sorted(relevance, reverse=True)
    idcg = dcg_at_k(ideal, k)
    return dcg_at_k(relevance, k) / idcg if idcg > 0 else 0.0

def mrr(relevance):
    for i, r in enumerate(relevance):
        if r == 1:
            return 1.0 / (i + 1)
    return 0.0

def evaluate(behaviors_path, score_fn):
    aucs, mrrs, ndcg5s, ndcg10s = [], [], [], []
    skipped = 0

    with open(behaviors_path, encoding="utf-8") as f:
        for line in f:
            cols = line.strip().split("\t")
            if len(cols) < 5:
                continue

            history    = cols[3].strip().split() if cols[3].strip() else []
            impressions = cols[4].strip().split()

            candidates, labels = [], []
            for imp in impressions:
                parts = imp.rsplit("-", 1)
                if len(parts) != 2:
                    continue
                nid, label = parts
                candidates.append(nid)
                labels.append(int(label))

            if sum(labels) == 0 or sum(labels) == len(labels):
                skipped += 1
                continue

            scores = score_fn(history, candidates)
            ranked = [lbl for _, lbl in sorted(zip(scores, labels), reverse=True)]

            aucs.append(roc_auc_score(labels, scores))
            mrrs.append(mrr(ranked))
            ndcg5s.append(ndcg_at_k(ranked, 5))
            ndcg10s.append(ndcg_at_k(ranked, 10))

    print(f"Evaluated {len(aucs)} impressions | Skipped (degenerate): {skipped}")
    return {
        "AUC":     round(float(np.mean(aucs)), 4),
        "MRR":     round(float(np.mean(mrrs)), 4),
        "nDCG@5":  round(float(np.mean(ndcg5s)), 4),
        "nDCG@10": round(float(np.mean(ndcg10s)), 4),
    }

In [7]:
metrics = evaluate(DEV_BEHAVIORS, cf_score)

print("\nItem-Based CF — Dev Set Results")
print("-" * 40)
for k, v in metrics.items():
    print(f"  {k:<10} {v}")

Evaluated 73152 impressions | Skipped (degenerate): 0

Item-Based CF — Dev Set Results
----------------------------------------
  AUC        0.5209
  MRR        0.3671
  nDCG@5     0.3679
  nDCG@10    0.4308


**What these numbers mean:**

Compare these against the baseline (AUC 0.5318, MRR 0.3387, nDCG@5 0.3321, nDCG@10 0.3982). If item-based CF is working, you should see a meaningful improvement — especially on AUC, because the model is now actually looking at each user's personal history instead of recommending the same popular articles to everyone.

  The AUC vs ranking metrics split is actually interesting, not a failure.                                                                                 - AUC looks at all candidates — clicked vs non-clicked across the whole list
  - MRR and nDCG only care about the top of the ranking                                                                                                  
  - The CF is doing a better job pushing clicked articles to the top (better MRR/nDCG), but probably making confident wrong calls on cold-start
  candidates (articles it's never seen get score 0, which might accidentally rank a clicked cold-start article below non-clicked ones and hurt AUC)

## Step 6 — Reflection

Strengths of this approach:
- **Personalised** — rankings depend on each user's click history
- **No training loop** — the "model" is just the normalised interaction matrix
- **Interpretable** — for any recommendation we can point at which history articles caused the score

Limitations:
- **Cold-start articles**: any candidate not seen in training gets score 0, same problem as the baseline
- **Cold-start users**: a user with empty history gets all-zero scores, so ranking is arbitrary
- **Popularity bias**: articles that co-occur with many things will have high average similarity and drift to the top
- **Ignores content**: two articles about the same topic will look dissimilar if they were clicked by different user groups